# L6a: Introduction to Classical Hopfield Networks
In the next few lectures, we will study Artificial Neural Networks (ANNs), starting with classical Hopfield networks and progressing to modern Hopfield networks, Boltzmann machines, and deep neural networks. These models are computational systems for pattern recognition and function approximation.

We start with Hopfield networks to establish core neural-network concepts. [John Hopfield shared the Nobel Prize in Physics in 2024 for work related to neural-network-based associative memory](https://www.nobelprize.org/prizes/physics/2024/hopfield/facts/).

> __Learning Objectives:__
> 
> By the end of this lecture, you should be able to define and explain:
>
> * __McCulloch-Pitts Neurons:__ Describe the foundational McCulloch-Pitts neuron model with binary inputs and outputs, understand how the weighted sum of inputs is compared against a threshold using an activation function, and recognize how this simple model underpins modern artificial neural networks including Hopfield networks.
> * __Classical Hopfield Networks:__ Define classical Hopfield networks as fully connected, recurrent neural networks with binary states, understand how memories are encoded using the Hebbian learning rule and the outer product of stored patterns, and explain how the energy function enables memory retrieval through asynchronous updates that guarantee convergence to stable patterns.
> * __Memory Retrieval and Convergence:__ Describe the asynchronous update algorithm where neurons are randomly updated to minimize the global energy function, understand convergence criteria including state stability and Hamming distance checks, and recognize limitations including finite storage capacity and convergence to spurious attractors or antipatterns.

Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Analyze a classical Hopfield Network](CHEME-5820-L6a-Example-Classical-HopfieldNetworks-Spring-2026.ipynb). In this example, we analyze an example of a classical Hopfield Network to understand how it encodes and retrieves binary patterns using Hebbian learning and asynchronous updates. We consider uncorrelated binary patterns and investigate the network's ability to recover original patterns from noisy inputs.

___

## Origin story: McCulloch-Pitts Neurons
In [their paper, McCulloch and Pitts (1943)](https://link.springer.com/article/10.1007/BF02478259), the authors studied how many [interconnected basic cells (neurons)](https://en.wikipedia.org/wiki/Biological_neuron_model) can generate structured behavior. They proposed a simplified neuron model that became a foundation for artificial neural networks.

<div>
  <center>
    <img
      src="figs/Fig-McCulloch-Pitts-Neuron-Schematic.svg"
      alt="McCulloch-Pitts neuron schematic"
      height="360"
      width="980"
    />
  </center>
</div>

Suppose we have a neuron that takes an input vector $\mathbf{n}(t) = (n^{(t)}_1, n^{(t)}_2, \ldots, n^{(t)}_{m})$, where each component $n_k\in\mathbf{n}$ is a binary value (`0` or `1`) representing the state of other predecessor neurons $n_1,n_2,\ldots,n_m$ at time $t$. Then, the state of our neuron (say neuron $k$) at time $t+1$ is given by:
$$
\begin{align*}
n_{k}(t+1) &= \sigma\left(\sum_{j=1}^{m} w_{kj} n_j(t) - b_k\right) \\
& = \sigma\left(\boldsymbol{\theta}_{k}^{\top} \hat{\mathbf{n}}(t)\right) \\
& = \sigma\left(\langle\boldsymbol{\theta}_{k}, \hat{\mathbf{n}}(t)\rangle\right)\quad\blacksquare
\end{align*}
$$
where $\sigma:\mathbb{R}\rightarrow\{0,1\}$ is an activation function that maps the weighted sum of inputs to a binary output, $\boldsymbol{\theta}_{k} = (w_{k1}, w_{k2}, \ldots, w_{km}, -b_k)$ is a vector of parameters (weights and bias) for neuron $k$, and $\hat{\mathbf{n}}(t) = (n_1(t), n_2(t), \ldots, n_m(t), 1)$ is the input vector augmented with a bias term.


In the original paper, the state of neuron $k$ at time $t+1$, denoted as $n_k(t+1)\in\{0,1\}$, where $w_{kj}$ is the weight of the connection from predecessor neuron $j$ to neuron $k$, and $b_k$ is the bias (threshold) for neuron $k$. 
* __Activation function__: In the original McCulloch and Pitts model, the activation function $\sigma$ is a step function, which means that the output of the neuron is `1` if the weighted sum of inputs exceeds the bias (threshold) value $b_k$, and `0` otherwise. In other words, the neuron "fires" (produces an output of `1`) if the total input to the neuron is greater than or equal to the bias (threshold) $b_k$. This is a __binary output__, simplifying real biological neurons that can produce continuous outputs.
* __Parameters__: The weights $w_{kj}\in\mathbb{R}$ and the bias $b_k\in\mathbb{R}$ are parameters of the neuron that determine its behavior. The weights can be positive or negative, representing the strength and direction of the influence of the input neurons on the output neuron. The bias determines how much input the neuron needs to "fire" (i.e., produce an output of `1`).

While the McCulloch-Pitts neuron model simplifies real biological neurons, it laid the groundwork for the development of more complex artificial neural networks. The key idea is that by combining many simple neurons in a network, we can create complex functions and learn to approximate any continuous function. This idea is at the heart of modern deep learning and neural networks. 

This model underpins later models including [the Perceptron (Rosenblatt, 1957)](https://en.wikipedia.org/wiki/Perceptron), [Hopfield networks](https://en.wikipedia.org/wiki/Hopfield_network), and [Boltzmann machines](https://en.wikipedia.org/wiki/Boltzmann_machine).

___

## Classical Hopfield Networks
A classical __Hopfield network__ is a fully connected, undirected graph $\mathcal{G} = \left(\mathcal{V}, \mathcal{E}\right)$ consisting of $\dim(\mathcal{V}) = N$ nodes (neurons, vertices, etc), where each node has a binary state $s = \pm 1$. Each node is connected to every other node, but not to itself, thus, we have $\dim(\mathcal{E}) = N(N-1)/2$ edges. The state of the network at time $t$ is represented by a vector $\mathbf{s}(t) = (s_1(t), s_2(t), \ldots, s_N(t))$, where each component $s_i(t) = \pm{1}$ is the state of node $i$ at time $t$. 

The connection weights between nodes $i$ and $j$, denoted $w_{ij} \in \mathbf{W}$, are determined using a **Hebbian learning rule**.

> __Hebbian learning__
>
> * __Hebbian Learning Rule__: The [Hebbian learning rule](https://en.wikipedia.org/wiki/Hebbian_theory), proposed by [Donald Hebb in 1949](https://en.wikipedia.org/wiki/Donald_O._Hebb), states that synaptic connections between neurons are strengthened when they activate (fire) simultaneously, forming the biological basis for __associative learning__. This "fire together, wire together" principle underpins unsupervised learning in neural networks, linking co-active nodes to enable pattern storage and recall.
> * __Different?__ Unlike the previous examples of learning, e.g., logistic regression or any of the online learning approaches that we looked at previously, the parameters (weights) in a [Hopfield network](https://en.wikipedia.org/wiki/Hopfield_network) are entirely specified by the memories we want to encode. Thus, we do not need to search for weights or learn them by experimenting with the world. Instead, we can directly compute the weights from the memories we want to encode.
> * __Recurrent?__ A Hopfield network is a special type of [recurrent neural network](https://en.wikipedia.org/wiki/Recurrent_neural_network) in which recurrence is used to settle into a stable pattern iteratively. It is considered recurrent because its units are symmetrically and recurrently connected, allowing the network to evolve toward an energy minimum over time.
> 
> The Hebbian learning rule uses only local computations without explicit training iterations, providing a biological basis for memory encoding that operates without specialized hardware or extended optimization cycles.


### Encoding memories into a Hopfield network
Suppose we wish our network to memorize $K$ images, where each image is an $n\times{n}$ collection of black and white pixels represented as a vector $\mathbf{s}_{i}\in\left\{-1,1\right\}^{n^2}$. We encode the image using the following rule: if the pixel is white, we set the memory value to `1`, and if the pixel is black, we set the memory value to `-1`. Then, the weights that encode these $K$ images are given by:
$$
\begin{align*}
\mathbf{W} &= \frac{1}{N}\sum_{i=1}^{K}\underbrace{\mathbf{s}_{i}\otimes\mathbf{s}_{i}^{\top}}_{\text{outer product}}\\
& = \frac{1}{N}\sum_{i=1}^{K}\mathbf{s}_{i}\mathbf{s}_{i}^{\top}\in\mathbb{R}^{n^2\times n^2}
\end{align*}
$$
where $\mathbf{s}_{i}$ denotes the state vector (pixels) of the image we want to memorize, $\otimes$ denotes the outer product, and $N=n^2$ is the number of neurons. Thus, each stored pattern contributes an outer-product term and the scale factor is set by network size.

> __How many memories can we store?__
> 
> The maximum theoretical storage limit $K_{\text{max}}$ of a classical Hopfield network, using the standard Hebbian learning rule, is approximately $K_{\text{max}}\sim 0.138N$, where $N$ is the number of neurons in the network. Thus, the network can reliably store about 14% of its size in patterns before retrieval errors become significant due to interference between stored patterns.

Suppose we've encoded $K$ images and want to retrieve one of them. How does retrieval work in practice?

### Algorithm: Memory retrieval
Each memory in a Hopfield network is encoded as a _local minimum_ of a global energy function. Thus, during memory retrieval, when we supply a random state vector $\hat{\mathbf{s}}$, we will recover the _closest_ memory encoded in the network to where we start, i.e., the memory that corresponds to the local minimum of the energy function that is closest to our initial state.

The overall energy of the network is given by:
$$
\begin{equation*}
E(\mathbf{s}) = -\frac{1}{2}\sum_{ij}w_{ij}s_{i}s_{j} - \sum_{i}b_{i}s_{i}\in\mathbb{R}
\end{equation*}
$$
where $w_{ij}\in\mathbf{W}$ are the weights of the network, $s_{i}\in\{-1,1\}$ are the states of the neuron $i$, and $b_{i}$ is a bias term (typically set to zero but can be used to control the activation threshold of the neurons). How do we go from a random state vector (memory) $\hat{\mathbf{s}}$ to a local minimum of the energy function $E(\mathbf{s})$? 

Let's outline some pseudocode for the memory retrieval algorithm.

__Initialize__: Compute the weights $w_{ij}\in\mathbf{W}$ using the Hebbian learning rule. Initialize the network with a random state $\mathbf{s}$. Set $\texttt{converged}\gets\texttt{false}$, the iteration counter $t\gets{1}$, maximum iterations $\texttt{maxiter} = 10N$ (where $N$ is the number of neurons), and patience parameter $\texttt{patience}\in\mathbb{Z}_{\geq 1}$.

> **Patience Parameter?** 
> 
> The patience parameter determines how many consecutive identical states are required to declare convergence. It is a practical heuristic that balances convergence detection with computational efficiency. Classical Hopfield networks can occasionally get stuck in short oscillation cycles (e.g., alternating between a few states). 
>
> __Fix:__ Requiring a fixed number of consecutive identical states ensures the network has truly converged to a stable attractor rather than just pausing briefly or terminating prematurely. 

__Track__: Initialize a queue $\texttt{S}$ to store the last $\texttt{patience}$ state vectors.

While not $\texttt{converged}$ __do__:
1. Store the current state: $\mathbf{s}_{\text{old}} \gets \mathbf{s}$.
2. **Asynchronous update**: Choose a random neuron (node) $i$ and compute a new state $s_{i}^{\prime}$ for neuron $i$ using the update rule: $s_{i}^{\prime} \leftarrow \texttt{sign}\left(\sum_{j}w_{ij}s_{j}+b_{i}\right)$, where $\texttt{sign}(\cdot)$ is the sign (activation) function, $b_{i}$ is a bias parameter (typically set to `0` for classical Hopfield networks), and ties are handled by keeping the current state when the argument is zero.
3. Update the network state: $\mathbf{s} \leftarrow \mathbf{s}^{\prime}$ (only neuron $i$ changes).
4. Add the current state to the history queue: $\texttt{S}\gets\texttt{S} \cup \{\mathbf{s}\}$.
5. **Check for convergence**: There are several criteria we can use to stop the iteration and determine if the network has converged:
   - _State stability_: If the state history $\texttt{S}$ contains $\texttt{patience}$ states and all states in the history are identical (Hamming distance = 0 between all consecutive pairs), then set $\texttt{converged}\gets\texttt{true}$, and return the current state $\mathbf{s}$ as the retrieved memory.
   - _Memory retrieval_: Alternatively, if the current state $\mathbf{s}$ exactly matches any stored memory pattern $\mathbf{s}_k$ (Hamming distance = 0), then set $\texttt{converged}\gets\texttt{true}$, and return $\mathbf{s}$ as the retrieved memory.
   - _Fixed-point check_: If every neuron already matches the tie-preserving update rule $s_i=\texttt{sign}(\sum_j w_{ij}s_j+b_i)$ (or remains unchanged when the local field is zero), then set $\texttt{converged}\gets\texttt{true}$ and return $\mathbf{s}$.
   - _Max iterations_: If $t \geq \texttt{maxiter}$, set $\texttt{converged}\gets\texttt{true}$ and $\texttt{warn}$ the caller that maximum iterations was reached without convergence.
6. If the length of the state history queue $\texttt{S}$ exceeds $\texttt{patience}$ length, remove the oldest state.
7. Update iteration counter: $t \leftarrow t + 1$.

> **Hamming Distance**: The Hamming distance between two binary vectors $\mathbf{a}$ and $\mathbf{b}$ is defined as $H(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{N} \mathbb{I}[a_i \neq b_i]$, where $\mathbb{I}[\cdot]$ is the indicator function. For convergence, we check if $H(\mathbf{s}_{\text{current}}, \mathbf{s}_{\text{previous}}) = 0$, meaning the states are identical.

### Convergence
Separate convergence into two questions: why asynchronous updates stop, and what kind of state they stop at.

> __Convergence Theorem (Classical Hopfield, asynchronous updates):__
>
> Assume symmetric weights $w_{ij}=w_{ji}$ and no self-connections $w_{ii}=0$. Define the local field and energy as:
> $$
> \begin{align*}
> h_i(\mathbf{s}) &= \sum_{j=1}^{N} w_{ij}s_j + b_i\\
> E(\mathbf{s}) &= -\frac{1}{2}\sum_{j=1}^{N}\sum_{k=1}^{N} w_{jk}s_js_k - \sum_{j=1}^{N} b_j s_j.
> \end{align*}
> $$
> If one asynchronous update changes only neuron $i$ with the tie-preserving rule $s_i' = \mathrm{sign}(h_i(\mathbf{s}))$ and $s_i'=s_i$ when $h_i(\mathbf{s})=0$, then the one-step energy change is given by:
> $$
> \Delta E = E(\mathbf{s}') - E(\mathbf{s}) = -(s_i' - s_i)h_i(\mathbf{s}) \le 0.
> $$
> If $s_i'\neq s_i$, then $\Delta E<0$. Because the state space has size $2^N$ and the energy is lower-bounded, repeated asynchronous updates reach a fixed point in finitely many accepted flips. For the full line-by-line derivation and the Hebbian signal-versus-crosstalk proof, see [▶ Advanced: Why Classical Hopfield Networks Converge and Why Hebbian Learning Works](CHEME-5820-L6a-Advanced-Classical-HopfieldConvergence-HebbianLearning-Spring-2026.ipynb).

The theorem guarantees that updates stop, but it does not guarantee retrieval of the intended memory.

- **What is guaranteed**: Energy descent guarantees that asynchronous updates stop at a stable fixed point instead of wandering indefinitely. This makes Hopfield retrieval computationally well-defined as a dynamical process.
- **What is not guaranteed**: The fixed point is not always the memory you intended to retrieve, especially at higher memory load or stronger cue corruption. Convergence answers "will the dynamics stop?" but not automatically "will it stop at the correct memory?"

Practical failure modes near capacity ($K\approx0.138N$) are listed below:
- **Spurious or incomplete attractors**: The network can settle into local minima that were not explicitly stored because of interference between patterns. These states can look like corrupted memory fragments, but asynchronous updates still treat them as stable endpoints.
- **Wrong stored memories**: If the initial state is near overlapping basins of attraction, the dynamics can converge to a different stored pattern than the intended one. This ambiguity is more likely when memories are correlated or when the cue is heavily corrupted by noise.
- **Antipatterns**: The network can converge to the bitwise inverse of a stored memory (for example, if $\mathbf{s}_k$ is stored, then $-\mathbf{s}_k$ can also be stable). This occurs because the Hebbian rule creates a symmetric energy landscape that can support both a pattern and its inverse.

As the number of stored patterns approaches the theoretical limit $K \approx 0.138N$, interference increases and these failure modes become more common. This is why convergence alone is not the same as correct retrieval.

Let's look at an example to illustrate these concepts.

> __Example:__
> 
> [▶ Analyze a classical Hopfield Network](CHEME-5820-L6a-Example-Classical-HopfieldNetworks-Spring-2026.ipynb). In this example, we analyze an example of a classical Hopfield Network to understand how it encodes and retrieves binary patterns using Hebbian learning and asynchronous updates. We consider uncorrelated binary patterns and investigate the network's ability to recover original patterns from noisy inputs.
____

## Lab
In the lab, we will implement a classical Hopfield network from scratch and analyze its behavior on a set of binary patterns. We will encode a few simple patterns into the network, introduce noise to the input, and observe how the network retrieves the original patterns through asynchronous updates.

## Summary

In this lecture, we explored classical Hopfield networks as associative memory systems:

> __Key takeaways:__
>
> * **Hebbian Learning and Memory Encoding**: Hopfield networks encode multiple binary patterns as local minima in an energy landscape using the Hebbian learning rule, computing weights from the outer product of stored patterns without requiring iterative training. This provides direct memory storage without explicit optimization.
> * **Guaranteed Convergence with Limitations**: Asynchronous updates monotonically decrease the network's energy function, guaranteeing convergence to stable states. However, convergence is limited by low storage capacity relative to network size, and the network may converge to spurious attractors, antipatterns, or incomplete memories rather than the intended stored pattern.
> * **Binary Associative Memory**: Classical Hopfield networks retrieve binary patterns from noisy or partial inputs by iteratively updating neurons based on the weighted sum of connected neurons. Memory retrieval is robust for small networks with few stored patterns but degrades significantly as the number of patterns approaches the theoretical capacity limit.

These points motivate the advanced derivation notebook, where we prove the energy-descent identity line by line and analyze when recall fails because of crosstalk.
___